# 01 - Data Understanding (Lending Club - Accepted Loans 2007-2018Q4)

Sondagem inicial de `accepted_2007_to_2018Q4.csv` (~1.67 GB, ~151 colunas, ~2.26M linhas).
Este notebook e somente leitura: nenhuma limpeza, exclusao ou transformacao e realizada aqui.

**Estrategia de memoria:** a maquina tem ~5.5 GB de RAM livre. O arquivo inteiro nao e carregado
de uma so vez. Uma amostra (`nrows=50000`) e usada para estrutura/dtypes; estatisticas que exigem
o arquivo inteiro (nulos, value_counts, datas) sao calculadas em um unico passe por chunks
(`chunksize=50000`), acumulando contadores e descartando cada chunk da memoria assim que processado.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

RAW_PATH = Path('..') / 'data' / 'raw' / 'accepted_2007_to_2018Q4.csv'
CHUNK_SIZE = 50000


In [2]:
sample = pd.read_csv(RAW_PATH, nrows=50000, low_memory=False)
print('Amostra carregada:', sample.shape)


Amostra carregada: (50000, 151)


## 1. Shape total (linhas reais do arquivo inteiro) e colunas com dtype

In [3]:
print(f'Colunas totais: {sample.shape[1]}')
print(f'(dtypes inferidos a partir da amostra de {sample.shape[0]:,} linhas)')


Colunas totais: 151
(dtypes inferidos a partir da amostra de 50,000 linhas)


In [4]:
print(sample.dtypes.to_string())


id                                              int64
member_id                                     float64
loan_amnt                                     float64
funded_amnt                                   float64
funded_amnt_inv                               float64
term                                              str
int_rate                                      float64
installment                                   float64
grade                                             str
sub_grade                                         str
emp_title                                         str
emp_length                                        str
home_ownership                                    str
annual_inc                                    float64
verification_status                               str
issue_d                                           str
loan_status                                       str
pymnt_plan                                        str
url                         

### Passe unico pelo arquivo inteiro (chunks de 50.000 linhas): conta linhas, nulos, loan_status e datas de issue_d

In [5]:
columns = sample.columns.tolist()
null_counts = pd.Series(0, index=columns, dtype='int64')
total_rows = 0
loan_status_counts = pd.Series(dtype='int64')
issue_d_min = None
issue_d_max = None
year_counts = pd.Series(dtype='int64')

for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    total_rows += len(chunk)
    null_counts = null_counts.add(chunk.isnull().sum(), fill_value=0)
    loan_status_counts = loan_status_counts.add(chunk['loan_status'].value_counts(), fill_value=0)

    issue_d_parsed = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    chunk_min = issue_d_parsed.min()
    chunk_max = issue_d_parsed.max()
    if pd.notnull(chunk_min) and (issue_d_min is None or chunk_min < issue_d_min):
        issue_d_min = chunk_min
    if pd.notnull(chunk_max) and (issue_d_max is None or chunk_max > issue_d_max):
        issue_d_max = chunk_max
    year_counts = year_counts.add(issue_d_parsed.dt.year.value_counts(), fill_value=0)

null_counts = null_counts.astype('int64')
loan_status_counts = loan_status_counts.astype('int64')
year_counts = year_counts.astype('int64').sort_index()

print(f'Total de linhas no arquivo inteiro: {total_rows:,}')


Total de linhas no arquivo inteiro: 2,260,701


In [6]:
print(f'Shape total (linhas x colunas): ({total_rows:,}, {sample.shape[1]})')


Shape total (linhas x colunas): (2,260,701, 151)


## 2. % de nulos por coluna (arquivo inteiro), ordenado do mais nulo pro menos

In [7]:
null_pct = (null_counts / total_rows * 100).sort_values(ascending=False).round(4)
null_pct_df = null_pct.to_frame('%_nulos')
null_pct_df['contagem_nulos'] = null_counts.reindex(null_pct_df.index)
null_pct_df


,%_nulos,contagem_nulos
member_id,100.0000,2260701
orig_projected_additional_accrued_interest,99.6173,2252050
hardship_reason,99.5171,2249784
hardship_payoff_balance_amount,99.5171,2249784
hardship_last_payment_amount,99.5171,2249784
payment_plan_start_date,99.5171,2249784
hardship_type,99.5171,2249784
hardship_status,99.5171,2249784
hardship_start_date,99.5171,2249784
deferral_term,99.5171,2249784


## 3. value_counts completo de loan_status (arquivo inteiro)

In [8]:
loan_status_pct = (loan_status_counts / total_rows * 100).round(4)
loan_status_df = pd.DataFrame({
    'contagem': loan_status_counts,
    '%': loan_status_pct
}).sort_values('contagem', ascending=False)
loan_status_df


,contagem,%
loan_status,,
Fully Paid,1076751,47.6291
Current,878317,38.8515
Charged Off,268559,11.8795
Late (31-120 days),21467,0.9496
In Grace Period,8436,0.3732
Late (16-30 days),4349,0.1924
Does not meet the credit policy. Status:Fully Paid,1988,0.0879
Does not meet the credit policy. Status:Charged Off,761,0.0337
Default,40,0.0018


## 4. Intervalo temporal de issue_d e contagem de emprestimos por ano

In [9]:
print('issue_d min:', issue_d_min)
print('issue_d max:', issue_d_max)


issue_d min: 2007-06-01 00:00:00
issue_d max: 2018-12-01 00:00:00


In [10]:
print('Emprestimos por ano de emissao:')
year_counts


Emprestimos por ano de emissao:


issue_d
2007.0       603
2008.0      2393
2009.0      5281
2010.0     12537
2011.0     21721
2012.0     53367
2013.0    134814
2014.0    235629
2015.0    421095
2016.0    434407
2017.0    443579
2018.0    495242
dtype: int64

## 5. Describe das numericas centrais (loan_amnt, int_rate, annual_inc, dti, installment)

In [11]:
numeric_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'installment']
cat_cols = ['grade', 'sub_grade', 'purpose', 'home_ownership', 'emp_length', 'addr_state', 'term']

subset = pd.read_csv(RAW_PATH, usecols=numeric_cols + cat_cols, low_memory=False)
print('Subset carregado (apenas colunas necessarias):', subset.shape)


Subset carregado (apenas colunas necessarias): (2260701, 12)


In [12]:
subset[numeric_cols].describe()


,loan_amnt,int_rate,annual_inc,dti,installment
count,2.260668e+06,2.260668e+06,2.260664e+06,2.258957e+06,2.260668e+06
mean,1.504693e+04,1.309283e+01,7.799243e+04,1.882420e+01,4.458068e+02
std,9.190245e+03,4.832138e+00,1.126962e+05,1.418333e+01,2.671735e+02
min,5.000000e+02,5.310000e+00,0.000000e+00,-1.000000e+00,4.930000e+00
25%,8.000000e+03,9.490000e+00,4.600000e+04,1.189000e+01,2.516500e+02
50%,1.290000e+04,1.262000e+01,6.500000e+04,1.784000e+01,3.779900e+02
75%,2.000000e+04,1.599000e+01,9.300000e+04,2.449000e+01,5.933200e+02
max,4.000000e+04,3.099000e+01,1.100000e+08,9.990000e+02,1.719830e+03


## 6. Cardinalidade (nunique) de colunas categoricas principais

In [13]:
for col in cat_cols:
    print(f'{col}: nunique = {subset[col].nunique()}')


grade: nunique = 7
sub_grade: nunique = 35
purpose: nunique = 14
home_ownership: nunique = 6
emp_length: nunique = 11
addr_state: nunique = 51
term: nunique = 2


## 7. Primeiras 3 linhas (todas as colunas)

In [14]:
pd.read_csv(RAW_PATH, nrows=3, low_memory=False)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=68407277,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.7,13.0,w,0.0,0.0,4421.723917,4421.72,3600.0,821.72,0.0,0.0,0.0,Jan-2019,122.67,NaN,Mar-2019,564.0,560.0,0.0,30.0,1.0,Individual,NaN,NaN,NaN,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=68355089,NaN,small_business,Business,577xx,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.2,38.0,w,0.0,0.0,25679.660000,25679.66,24700.0,979.66,0.0,0.0,0.0,Jun-2016,926.35,NaN,Mar-2019,699.0,695.0,0.0,NaN,1.0,Individual,NaN,NaN,NaN,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,NaN,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,246

## Sondagem 2 - populacao aprovada, estrutura temporal e anomalias

Continuacao da sondagem, ainda somente leitura. Nenhuma limpeza permanente e nenhum arquivo em
`data/processed` sao criados aqui. As definicoes de alvo e recorte de maturidade abaixo sao
aplicadas apenas em memoria, para fins de medicao. Nenhuma decisao de tratamento e tomada -
apenas evidencia.

### 1. Tabela cruzada: ano de issue_d x term x loan_status

In [15]:
core3 = pd.read_csv(RAW_PATH, usecols=['issue_d', 'term', 'loan_status'], low_memory=False)
core3['issue_dt'] = pd.to_datetime(core3['issue_d'], format='%b-%Y', errors='coerce')
core3['ano_emissao'] = core3['issue_dt'].dt.year
core3['term_clean'] = core3['term'].str.strip()
print('core3 carregado:', core3.shape)


core3 carregado: (2260701, 6)


In [16]:
crosstab_ano_term_status = pd.crosstab(
    [core3['ano_emissao'], core3['term_clean']],
    core3['loan_status'],
    dropna=False
)
crosstab_ano_term_status


loan_status             Charged Off  Current  Default  \
ano_emissao term_clean                                  
2007.0      36 months            45        0        0   
            60 months             0        0        0   
            nan                   0        0        0   
2008.0      36 months           247        0        0   
            60 months             0        0        0   
            nan                   0        0        0   
2009.0      36 months           594        0        0   
            60 months             0        0        0   
            nan                   0        0        0   
2010.0      36 months           842        0        0   
            60 months           645        0        0   
            nan                   0        0        0   
2011.0      36 months          1499        0        0   
            60 months          1798        0        0   
            nan                   0        0        0   
2012.0      36 months          5903        0        0   
            60 months          2741        0        0   
            nan                   0        0        0   
2013.0      36 months         12378        0        0   
            60 months          8646        6        0   
            nan                   0        0        0   
2014.0      36 months         22315        0        0   
            60 months         18846    11919        1   
            nan                   0        0        0   
2015.0      36 months         42132       72        0   
            60 months         33671    43227        1   
            nan                   0        0        0   
2016.0      36 months         46112    86623        8   
            60 months         22130    47438        2   
            nan                   0        0        0   
2017.0      36 months         25721   183950       13   
            60 months         13427    77901        8   
            nan                   0        0        0   
2018.0      36 months          5464   296050        4   
            60 months          3403   131131        3   
            nan                   0        0        0   
NaN         36 months             0        0        0   
            60 months             0        0        0   
            nan                   0        0        0   

loan_status             Does not meet the credit policy. Status:Charged Off  \
ano_emissao term_clean                                                        
2007.0      36 months                                                   113   
            60 months                                                     0   
            nan                                                           0   
2008.0      36 months                                                   249   
            60 months                                                     0   
            nan                                                           0   
2009.0      36 months                                                   129   
            60 months                                                     0   
            nan                                                           0   
2010.0      36 months                                                   158   
            60 months                                                   112   
            nan                                                           0   
2011.0      36 months                                                     0   
            60 months                                                     0   
            nan                                                           0   
2012.0      36 months                                                     0   
            60 months                                                     0   
            nan                                                           0   
2013.0      36 months                                                     0   
   

### 2. Populacao analitica aprovada

**Definicoes aplicadas (em memoria, apenas para medir):**
- Alvo: `Fully Paid` = 0, `Charged Off` = 1. Todos os demais status ficam fora da populacao.
- Recorte de maturidade: term de 36 meses -> issue_d ate dez/2015; term de 60 meses -> issue_d ate dez/2013.

In [17]:
TARGET_MAP = {'Fully Paid': 0, 'Charged Off': 1}
CUTOFF_36 = pd.Timestamp('2015-12-01')
CUTOFF_60 = pd.Timestamp('2013-12-01')

is_target_status = core3['loan_status'].isin(['Fully Paid', 'Charged Off'])
status_excluded_counts = core3.loc[~is_target_status, 'loan_status'].value_counts(dropna=False)

kept = core3.loc[is_target_status].copy()
kept['target'] = kept['loan_status'].map(TARGET_MAP)

is_36 = kept['term_clean'] == '36 months'
is_60 = kept['term_clean'] == '60 months'
mature_36 = is_36 & kept['issue_dt'].notna() & (kept['issue_dt'] <= CUTOFF_36)
mature_60 = is_60 & kept['issue_dt'].notna() & (kept['issue_dt'] <= CUTOFF_60)
mature_mask = mature_36 | mature_60

immature_36_count = int((is_36 & ~mature_36).sum())
immature_60_count = int((is_60 & ~mature_60).sum())
other_term_count = int((~is_36 & ~is_60).sum())

final_pop = kept.loc[mature_mask].copy()
n_final = len(final_pop)
default_rate = final_pop['target'].mean() * 100

print(f'N final da populacao analitica: {n_final:,}')
print(f'% de default (Charged Off / total): {default_rate:.4f}%')


N final da populacao analitica: 673,553
% de default (Charged Off / total): 14.8147%


#### 2.b % de default por ano de emissao x term, dentro da populacao

In [18]:
default_by_vintage = final_pop.groupby(['ano_emissao', 'term_clean'])['target'].agg(n='count', pct_default='mean')
default_by_vintage['pct_default'] = (default_by_vintage['pct_default'] * 100).round(4)
default_by_vintage


n  pct_default
ano_emissao term_clean                     
2007.0      36 months      251      17.9283
2008.0      36 months     1562      15.8131
2009.0      36 months     4716      12.5954
2010.0      36 months     8466       9.9457
            60 months     3070      21.0098
2011.0      36 months    14101      10.6305
            60 months     7620      23.5958
2012.0      36 months    43470      13.5795
            60 months     9897      27.6953
2013.0      36 months   100422      12.3260
            60 months    34382      25.1469
2014.0      36 months   162570      13.7264
2015.0      36 months   283026      14.8863

#### 2.c Funil de exclusoes (rastreabilidade total do arquivo inteiro ate N final)

In [19]:
funnel_rows = [('Total no arquivo', total_rows)]
for status_val, cnt in status_excluded_counts.items():
    label = status_val if pd.notnull(status_val) else 'loan_status ausente (NaN)'
    funnel_rows.append((f'Excluido - status = "{label}"', int(cnt)))
funnel_rows.append(('Excluido - term ausente/invalido (nem 36 nem 60 meses)', other_term_count))
funnel_rows.append(('Excluido - safra imatura (term 36 meses, issue_d apos dez/2015)', immature_36_count))
funnel_rows.append(('Excluido - safra imatura (term 60 meses, issue_d apos dez/2013)', immature_60_count))
funnel_rows.append(('N final - populacao analitica', n_final))

funnel_df = pd.DataFrame(funnel_rows, columns=['motivo', 'contagem'])
funnel_df['%_do_total'] = (funnel_df['contagem'] / total_rows * 100).round(4)

soma_exclusoes_e_final = funnel_df.loc[funnel_df['motivo'] != 'Total no arquivo', 'contagem'].sum()
print(f'Checagem: total_rows ({total_rows:,}) == soma das exclusoes + N final ({soma_exclusoes_e_final:,})? '
      f'{total_rows == soma_exclusoes_e_final}')

funnel_df


Checagem: total_rows (2,260,701) == soma das exclusoes + N final (2,260,701)? True


,motivo,contagem,%_do_total
0,Total no arquivo,2260701,100.0000
1,"Excluido - status = ""Current""",878317,38.8515
2,"Excluido - status = ""Late (31-120 days)""",21467,0.9496
3,"Excluido - status = ""In Grace Period""",8436,0.3732
4,"Excluido - status = ""Late (16-30 days)""",4349,0.1924
5,"Excluido - status = ""Does not meet the credit policy. Status:Fully Paid""",1988,0.0879
6,"Excluido - status = ""Does not meet the credit policy. Status:Charged Off""",761,0.0337
7,"Excluido - status = ""Default""",40,0.0018
8,"Excluido - status = ""loan_status ausente (NaN)""",33,0.0015
9,Excluido - term ausente/invalido (nem 36 nem 60 meses),0,0.0000


### 3. Passe unico full-columns (chunks de 50.000 linhas)

Calcula, em um unico passe pelo arquivo inteiro: nulos por coluna dentro da populacao aprovada,
linhas com `loan_amnt` nulo (ancora das colunas centrais), anomalias de `dti` e de `annual_inc`.

In [20]:
pop_null_counts = pd.Series(0, index=columns, dtype='int64')
pop_total_rows = 0

core_null_rows_list = []

dti_neg_file_count = 0
dti_gt100_file_count = 0
dti_neg_pop_count = 0
dti_gt100_pop_count = 0
dti_neg_examples_list = []
dti_neg_examples_rows = 0
dti_gt100_examples_list = []
dti_gt100_examples_rows = 0

annual_inc_zero_count = 0
annual_inc_null_count = 0
top20_cols = ['id', 'annual_inc', 'loan_amnt', 'grade', 'home_ownership', 'loan_status']
running_top20 = None

for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    status_ok = chunk['loan_status'].isin(['Fully Paid', 'Charged Off'])
    issue_dt_chunk = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    term_clean_chunk = chunk['term'].str.strip()
    is_36_chunk = term_clean_chunk == '36 months'
    is_60_chunk = term_clean_chunk == '60 months'
    mature_chunk = (
        (is_36_chunk & issue_dt_chunk.notna() & (issue_dt_chunk <= CUTOFF_36)) |
        (is_60_chunk & issue_dt_chunk.notna() & (issue_dt_chunk <= CUTOFF_60))
    )
    pop_mask = status_ok & mature_chunk

    pop_total_rows += int(pop_mask.sum())
    pop_null_counts = pop_null_counts.add(chunk.loc[pop_mask].isnull().sum(), fill_value=0)

    core_null_mask = chunk['loan_amnt'].isnull()
    if core_null_mask.any():
        core_null_rows_list.append(chunk.loc[core_null_mask])

    dti_neg_mask = chunk['dti'] < 0
    dti_gt100_mask = chunk['dti'] > 100
    dti_neg_file_count += int(dti_neg_mask.sum())
    dti_gt100_file_count += int(dti_gt100_mask.sum())
    dti_neg_pop_count += int((dti_neg_mask & pop_mask).sum())
    dti_gt100_pop_count += int((dti_gt100_mask & pop_mask).sum())

    if dti_neg_examples_rows < 5 and dti_neg_mask.any():
        take = chunk.loc[dti_neg_mask, ['loan_amnt', 'annual_inc', 'dti', 'loan_status']].head(5 - dti_neg_examples_rows)
        dti_neg_examples_list.append(take)
        dti_neg_examples_rows += len(take)

    if dti_gt100_examples_rows < 5 and dti_gt100_mask.any():
        take = chunk.loc[dti_gt100_mask, ['loan_amnt', 'annual_inc', 'dti', 'loan_status']].head(5 - dti_gt100_examples_rows)
        dti_gt100_examples_list.append(take)
        dti_gt100_examples_rows += len(take)

    annual_inc_zero_count += int((chunk['annual_inc'] == 0).sum())
    annual_inc_null_count += int(chunk['annual_inc'].isnull().sum())

    chunk_top20 = chunk.loc[chunk['annual_inc'].notna(), top20_cols].nlargest(20, 'annual_inc')
    running_top20 = chunk_top20 if running_top20 is None else pd.concat([running_top20, chunk_top20], ignore_index=True).nlargest(20, 'annual_inc')

pop_null_counts = pop_null_counts.astype('int64')
core_null_rows = pd.concat(core_null_rows_list, ignore_index=True) if core_null_rows_list else pd.DataFrame(columns=columns)
dti_neg_examples_df = pd.concat(dti_neg_examples_list, ignore_index=True) if dti_neg_examples_list else pd.DataFrame()
dti_gt100_examples_df = pd.concat(dti_gt100_examples_list, ignore_index=True) if dti_gt100_examples_list else pd.DataFrame()

print(f'pop_total_rows (passe full-columns) = {pop_total_rows:,}')
print(f'n_final (passe subset issue_d/term/loan_status) = {n_final:,}')
print(f'Consistentes? {pop_total_rows == n_final}')


pop_total_rows (passe full-columns) = 673,553
n_final (passe subset issue_d/term/loan_status) = 673,553
Consistentes? True


#### Inventario de colunas -> docs/column_inventory.csv

In [21]:
null_pct_file_all = (null_counts / total_rows * 100).round(4)
null_pct_pop_all = (pop_null_counts / pop_total_rows * 100).round(4)

column_inventory = pd.DataFrame({
    'nome': columns,
    'dtype': [str(sample.dtypes[c]) for c in columns],
    '%nulos_arquivo_inteiro': [null_pct_file_all[c] for c in columns],
    '%nulos_populacao_aprovada': [null_pct_pop_all[c] for c in columns],
    'suspeita_vazamento': ['' for _ in columns],
    'nota': ['' for _ in columns],
})

docs_path = Path('..') / 'docs' / 'column_inventory.csv'
column_inventory.to_csv(docs_path, index=False)
print('Salvo em:', docs_path.resolve())
print('Shape:', column_inventory.shape)
column_inventory.head(10)


Salvo em: C:\Users\Avell\Documents\Projetos\credit-default-prediction-lendingclub\docs\column_inventory.csv
Shape: (151, 6)


,nome,dtype,%nulos_arquivo_inteiro,%nulos_populacao_aprovada,suspeita_vazamento,nota
0,id,int64,0.0000,0.0,,
1,member_id,float64,100.0000,100.0,,
2,loan_amnt,float64,0.0015,0.0,,
3,funded_amnt,float64,0.0015,0.0,,
4,funded_amnt_inv,float64,0.0015,0.0,,
5,term,str,0.0015,0.0,,
6,int_rate,float64,0.0015,0.0,,
7,installment,float64,0.0015,0.0,,
8,grade,str,0.0015,0.0,,
9,sub_grade,str,0.0015,0.0,,


### 4. As linhas com loan_amnt nulo (ancora das colunas centrais)

In [22]:
print(f'Linhas com loan_amnt nulo: {len(core_null_rows)}')
non_null_counts_per_row = core_null_rows.notnull().sum(axis=1)
print('Campos preenchidos por linha (maximo possivel = 151):')
print(non_null_counts_per_row.describe())


Linhas com loan_amnt nulo: 33
Campos preenchidos por linha (maximo possivel = 151):
count    33.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64


In [23]:
core_null_rows


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,Total amount funded in policy code 1: 6417608175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Total amount funded in policy code 2: 1944088810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Total amount funded in policy code 1: 1741781700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

### 5. Anomalia dti: dti < 0 e dti > 100

In [24]:
print(f'dti < 0  - contagem no arquivo inteiro: {dti_neg_file_count:,}')
print(f'dti < 0  - contagem na populacao aprovada:  {dti_neg_pop_count:,}')
print(f'dti > 100 - contagem no arquivo inteiro: {dti_gt100_file_count:,}')
print(f'dti > 100 - contagem na populacao aprovada:  {dti_gt100_pop_count:,}')


dti < 0  - contagem no arquivo inteiro: 2
dti < 0  - contagem na populacao aprovada:  0
dti > 100 - contagem no arquivo inteiro: 2,561
dti > 100 - contagem na populacao aprovada:  5


In [25]:
print('5 exemplos de dti < 0:')
dti_neg_examples_df


5 exemplos de dti < 0:


,loan_amnt,annual_inc,dti,loan_status
0,15000.0,94000.0,-1.0,Fully Paid
1,17000.0,75000.0,-1.0,Fully Paid


In [26]:
print('5 exemplos de dti > 100:')
dti_gt100_examples_df


5 exemplos de dti > 100:


,loan_amnt,annual_inc,dti,loan_status
0,6550.0,1770.0,999.00,Fully Paid
1,15000.0,20000.0,137.40,Fully Paid
2,28000.0,17000.0,136.97,Current
3,20000.0,27000.0,100.09,Fully Paid
4,15000.0,8700.0,120.66,Fully Paid


### 6. Anomalia annual_inc: maiores rendas, zeros e nulos

In [27]:
print('20 maiores annual_inc (arquivo inteiro):')
running_top20.sort_values('annual_inc', ascending=False).reset_index(drop=True)


20 maiores annual_inc (arquivo inteiro):

,id,annual_inc,loan_amnt,grade,home_ownership,loan_status
0,113245115,110000000.0,30000.0,B,RENT,Current
1,101074293,61000000.0,10000.0,B,MORTGAGE,Current
2,118198788,10999200.0,5000.0,D,RENT,Fully Paid
3,131421578,9930475.0,16000.0,C,MORTGAGE,Current
4,143613674,9757200.0,14000.0,C,MORTGAGE,Current
5,76323387,9573072.0,30000.0,A,OWN,Late (31-120 days)
6,74682106,9550000.0,30000.0,A,MORTGAGE,Fully Paid
7,99577890,9522972.0,35000.0,C,MORTGAGE,Fully Paid
8,54067210,9500000.0,24000.0,A,MORTGAGE,Charged Off
9,103851537,9300086.0,24000.0,B,MORTGAGE,Current


In [28]:
print(f'annual_inc == 0 - contagem: {annual_inc_zero_count:,}')
print(f'annual_inc nulo - contagem: {annual_inc_null_count:,}')


annual_inc == 0 - contagem: 1,667
annual_inc nulo - contagem: 37


## Sondagem 2b - verificacoes pendentes

Continuacao da sondagem, ainda somente leitura. Nenhuma decisao de tratamento e tomada aqui -
apenas evidencia, reaplicando em memoria as mesmas definicoes de populacao aprovada da Sondagem 2
(alvo Fully Paid=0/Charged Off=1; maturidade: 36 meses ate dez/2015, 60 meses ate dez/2013).

In [29]:
extra_cols = ['id', 'issue_d', 'term', 'loan_status', 'annual_inc', 'loan_amnt', 'grade',
              'home_ownership', 'emp_length', 'dti', 'purpose']
core_extra = pd.read_csv(RAW_PATH, usecols=extra_cols, low_memory=False)
core_extra['issue_dt'] = pd.to_datetime(core_extra['issue_d'], format='%b-%Y', errors='coerce')
core_extra['term_clean'] = core_extra['term'].str.strip()

is_target_status_e = core_extra['loan_status'].isin(['Fully Paid', 'Charged Off'])
is_36_e = core_extra['term_clean'] == '36 months'
is_60_e = core_extra['term_clean'] == '60 months'
mature_e = (
    (is_36_e & core_extra['issue_dt'].notna() & (core_extra['issue_dt'] <= CUTOFF_36)) |
    (is_60_e & core_extra['issue_dt'].notna() & (core_extra['issue_dt'] <= CUTOFF_60))
)
pop_mask_e = is_target_status_e & mature_e
pop_extra = core_extra.loc[pop_mask_e].copy()

print(f'pop_extra: {len(pop_extra):,} linhas (esperado = n_final = {n_final:,}) -> bate? {len(pop_extra) == n_final}')


pop_extra: 673,553 linhas (esperado = n_final = 673,553) -> bate? True


### 1. Complemento annual_inc: outliers de renda dentro da populacao aprovada

In [30]:
high_income_pop = pop_extra.loc[
    pop_extra['annual_inc'] > 1_000_000,
    ['annual_inc', 'loan_amnt', 'grade', 'home_ownership', 'emp_length', 'loan_status', 'dti']
].sort_values('annual_inc', ascending=False)

print(f'Linhas na populacao aprovada com annual_inc > 1.000.000: {len(high_income_pop)}')
high_income_pop


Linhas na populacao aprovada com annual_inc > 1.000.000: 120


,annual_inc,loan_amnt,grade,home_ownership,emp_length,loan_status,dti
40533,9000000.0,11000.0,A,MORTGAGE,10+ years,Fully Paid,0.08
217721,8900060.0,10550.0,D,RENT,10+ years,Charged Off,0.09
400638,8706582.0,8000.0,C,MORTGAGE,10+ years,Charged Off,0.11
48075,8500021.0,12000.0,B,MORTGAGE,10+ years,Fully Paid,0.22
85757,8253000.0,30000.0,C,OWN,10+ years,Fully Paid,0.14
128385,8121180.0,5000.0,B,MORTGAGE,10+ years,Fully Paid,0.48
132773,7600000.0,10000.0,A,MORTGAGE,3 years,Fully Paid,0.09
1211606,7500000.0,15000.0,E,MORTGAGE,10+ years,Charged Off,0.20
1338390,7446395.0,20000.0,A,RENT,5 years,Fully Paid,0.13
1885904,7141778.0,14825.0,B,MORTGAGE,10+ years,Fully Paid,0.25


In [31]:
top20_ids = set(running_top20['id'])
pop_ids = set(pop_extra['id'])
top20_in_pop_ids = top20_ids & pop_ids

print(f'Dos 20 maiores annual_inc do arquivo inteiro (Sondagem 2), quantos pertencem a populacao aprovada: {len(top20_in_pop_ids)}')
if top20_in_pop_ids:
    print(running_top20[running_top20['id'].isin(top20_in_pop_ids)])
else:
    print('Nenhum dos 20 maiores do arquivo inteiro pertence a populacao aprovada.')


Dos 20 maiores annual_inc do arquivo inteiro (Sondagem 2), quantos pertencem a populacao aprovada: 1
          id  annual_inc  loan_amnt grade home_ownership  loan_status
17  39429853   8706582.0     8000.0     C       MORTGAGE  Charged Off


### 2. Os casos de dti > 100 dentro da populacao aprovada

In [32]:
dti_gt100_in_pop = pop_extra.loc[
    pop_extra['dti'] > 100,
    ['loan_amnt', 'annual_inc', 'dti', 'emp_length', 'home_ownership', 'purpose', 'grade', 'issue_d', 'term_clean', 'loan_status']
].rename(columns={'term_clean': 'term'})

print(f'Casos de dti > 100 na populacao aprovada: {len(dti_gt100_in_pop)}')
dti_gt100_in_pop


Casos de dti > 100 na populacao aprovada: 5


,loan_amnt,annual_inc,dti,emp_length,home_ownership,purpose,grade,issue_d,term,loan_status
8721,6550.0,1770.0,999.00,NaN,MORTGAGE,credit_card,D,Dec-2015,36 months,Fully Paid
10548,15000.0,20000.0,137.40,NaN,MORTGAGE,debt_consolidation,D,Dec-2015,36 months,Fully Paid
55648,15000.0,8700.0,120.66,NaN,MORTGAGE,debt_consolidation,D,Dec-2015,36 months,Fully Paid
73833,12000.0,1200.0,672.52,NaN,RENT,debt_consolidation,E,Nov-2015,36 months,Fully Paid
97333,5000.0,12000.0,104.00,5 years,MORTGAGE,home_improvement,C,Oct-2015,36 months,Fully Paid
